In [3]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
import os
os.chdir('/content/drive/MyDrive/Mémoire_Master_Dataset')

In [5]:
"""
Génération des figures du mémoire
Détection robuste d'anomalies face aux intrusions adverses réseaux
Jemima SENKERE AFIDOU — Master 2 IA & Data Science, UNIKIN

Prérequis : pip install pandas matplotlib numpy
Placer ce script dans le même dossier que les 4 fichiers CSV.
Les figures seront sauvegardées dans un sous-dossier "figures/".
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ---------------------------------------------------------------------------
# Configuration générale
# ---------------------------------------------------------------------------

os.makedirs("figures", exist_ok=True)

# Palette sobre (compatible thèse imprimée)
COULEUR_BASELINE  = "#2C5F8A"   # bleu foncé
COULEUR_DEFENSE_A = "#C0392B"   # rouge
COULEUR_DEFENSE_B = "#27AE60"   # vert

COULEUR_SANS_ATT  = "#2C5F8A"
COULEUR_FGSM      = "#E67E22"
COULEUR_PGD       = "#C0392B"

plt.rcParams.update({
    "font.family":      "DejaVu Sans",
    "font.size":        11,
    "axes.titlesize":   12,
    "axes.labelsize":   11,
    "legend.fontsize":  10,
    "figure.dpi":       150,
    "axes.spines.top":  False,
    "axes.spines.right":False,
})

# ---------------------------------------------------------------------------
# FIGURE 1 — Vulnérabilité du modèle de base (resultats_graines_global.csv)
# ---------------------------------------------------------------------------
# Rappel moyen par condition d'attaque sur les 10 graines

df_global = pd.read_csv("resultats_graines_global.csv")

conditions = ["rappel_propre", "rappel_fgsm_0.02", "rappel_fgsm_0.05",
              "rappel_pgd_0.02", "rappel_pgd_0.05", "rappel_pgd_0.1"]
etiquettes = ["Sans attaque", "FGSM ε=0,02", "FGSM ε=0,05",
              "PGD ε=0,02",  "PGD ε=0,05",  "PGD ε=0,10"]

moyennes = [df_global[c].mean() for c in conditions]
ecarts   = [df_global[c].std()  for c in conditions]

couleurs_fig1 = [COULEUR_SANS_ATT,
                 COULEUR_FGSM, COULEUR_FGSM,
                 COULEUR_PGD,  COULEUR_PGD,  COULEUR_PGD]

fig, ax = plt.subplots(figsize=(8, 5))
barres = ax.bar(etiquettes, moyennes, yerr=ecarts, capsize=4,
                color=couleurs_fig1, alpha=0.85, width=0.6,
                error_kw={"elinewidth": 1.2, "ecolor": "#555"})

ax.set_ylim(0, 1.05)
ax.set_ylabel("Rappel moyen (10 graines)")
ax.set_title("Vulnérabilité du modèle de base face aux attaques adversariales")
ax.axhline(moyennes[0], color="#999", linewidth=0.8, linestyle="--")

for barre, val in zip(barres, moyennes):
    ax.text(barre.get_x() + barre.get_width() / 2, val + 0.02,
            f"{val:.2f}", ha="center", va="bottom", fontsize=9)

patch_fgsm = mpatches.Patch(color=COULEUR_FGSM, label="FGSM")
patch_pgd  = mpatches.Patch(color=COULEUR_PGD,  label="PGD")
patch_base = mpatches.Patch(color=COULEUR_SANS_ATT, label="Sans attaque")
ax.legend(handles=[patch_base, patch_fgsm, patch_pgd], loc="upper right")

plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("figures/fig1_vulnerabilite_baseline.png", bbox_inches="tight")
plt.close()
print("Figure 1 sauvegardée.")

# ---------------------------------------------------------------------------
# FIGURE 2 — Comparaison des défenses (resultats_defense.csv)
# ---------------------------------------------------------------------------
# Rappel propre et rappel sous PGD ε=0,05 pour baseline / A / B

df_def = pd.read_csv("resultats_defense.csv")

modeles = ["baseline", "defense_A", "defense_B"]
labels  = ["Baseline", "Défense A", "Défense B"]
couleurs_def = [COULEUR_BASELINE, COULEUR_DEFENSE_A, COULEUR_DEFENSE_B]

rappels_propres = []
rappels_pgd     = []
for m in modeles:
    sub = df_def[df_def["modele"] == m]
    rappels_propres.append((sub["rappel_propre"].mean(), sub["rappel_propre"].std()))
    rappels_pgd.append((sub["rappel_pgd_0.05"].mean(), sub["rappel_pgd_0.05"].std()))

x = np.arange(len(modeles))
largeur = 0.35

fig, ax = plt.subplots(figsize=(8, 5))

b1 = ax.bar(x - largeur/2,
            [r[0] for r in rappels_propres],
            largeur,
            yerr=[r[1] for r in rappels_propres],
            capsize=4,
            color=couleurs_def, alpha=0.9,
            error_kw={"elinewidth": 1.2, "ecolor": "#555"},
            label="Rappel propre")

b2 = ax.bar(x + largeur/2,
            [r[0] for r in rappels_pgd],
            largeur,
            yerr=[r[1] for r in rappels_pgd],
            capsize=4,
            color=couleurs_def, alpha=0.45,
            error_kw={"elinewidth": 1.2, "ecolor": "#555"},
            label="Rappel sous PGD ε=0,05")

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Rappel moyen (10 graines)")
ax.set_title("Comparaison des trois conditions : rappel propre et sous PGD (ε=0,05)")

# Valeurs au-dessus des barres
for barre in list(b1) + list(b2):
    h = barre.get_height()
    ax.text(barre.get_x() + barre.get_width()/2, h + 0.02,
            f"{h:.2f}", ha="center", va="bottom", fontsize=8.5)

# Légende : hachures pour distinguer propre / PGD
patch_propre = mpatches.Patch(facecolor="#888", alpha=0.9, label="Rappel propre (plein)")
patch_pgd_l  = mpatches.Patch(facecolor="#888", alpha=0.45, label="Rappel sous PGD (transparent)")
ax.legend(handles=[patch_propre, patch_pgd_l], loc="upper left")

plt.tight_layout()
plt.savefig("figures/fig2_comparaison_defenses.png", bbox_inches="tight")
plt.close()
print("Figure 2 sauvegardée.")

# ---------------------------------------------------------------------------
# FIGURE 3 — Rappel par famille d'attaque (resultats_defense_types.csv)
# ---------------------------------------------------------------------------
# Défense B vs Baseline, rappel propre, par famille

df_types = pd.read_csv("resultats_defense_types.csv")

familles_ordre = ["DoS GoldenEye", "DoS Hulk", "DoS Slowhttptest", "DoS slowloris",
                  "DDoS", "PortScan", "FTP - Patator", "SSH - Patator",
                  "Bot", "Web Attack - Brute Force", "Web Attack - XSS",
                  "Web Attack - Sql Injection", "Heartbleed", "Infiltration"]

def rappel_famille(df, modele, col="rappel_propre"):
    sub = df[df["modele"] == modele]
    return sub.groupby("type")[col].mean()

baseline_propre  = rappel_famille(df_types, "baseline",  "rappel_propre")
defB_propre      = rappel_famille(df_types, "defense_B", "rappel_propre")
baseline_pgd     = rappel_famille(df_types, "baseline",  "rappel_pgd")
defB_pgd         = rappel_famille(df_types, "defense_B", "rappel_pgd")

# Garder seulement les familles présentes dans les deux
familles = [f for f in familles_ordre if f in baseline_propre.index and f in defB_propre.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

x = np.arange(len(familles))
largeur = 0.4

for ax, col_base, col_def, titre in [
    (axes[0], baseline_propre, defB_propre,  "Rappel propre (sans attaque)"),
    (axes[1], baseline_pgd,    defB_pgd,     "Rappel sous PGD (ε=0,05)")
]:
    vals_base = [col_base.get(f, 0) for f in familles]
    vals_def  = [col_def.get(f, 0)  for f in familles]

    ax.barh(x + largeur/2, vals_base, largeur, color=COULEUR_BASELINE,
            alpha=0.85, label="Baseline")
    ax.barh(x - largeur/2, vals_def,  largeur, color=COULEUR_DEFENSE_B,
            alpha=0.85, label="Défense B")

    ax.set_xlim(0, 1.1)
    ax.set_title(titre)
    ax.set_xlabel("Rappel moyen")
    ax.axvline(0.5, color="#ccc", linewidth=0.8, linestyle="--")
    ax.legend(loc="lower right")

axes[0].set_yticks(x)
axes[0].set_yticklabels(familles, fontsize=9)

fig.suptitle("Rappel par famille d'attaque : Baseline vs Défense B", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("figures/fig3_rappel_par_famille.png", bbox_inches="tight")
plt.close()
print("Figure 3 sauvegardée.")

# ---------------------------------------------------------------------------
# FIGURE 4 — Généralisation leave-one-family-out (resultats_generalisation.csv)
# ---------------------------------------------------------------------------
# Rappel moyen sous PGD ε=0,05 par famille exclue, pour B_connue et B_inconnue

df_gen = pd.read_csv("resultats_generalisation.csv")

familles_gen = df_gen["famille"].unique()
B_connue   = df_gen[df_gen["modele"] == "B_famille_connue"]
B_inconnue = df_gen[df_gen["modele"] == "B_famille_inconnue"]
baseline_g = df_gen[df_gen["modele"] == "baseline"]

def moy_par_famille(df, col):
    return df.groupby("famille")[col].mean()

baseline_pgd_g  = moy_par_famille(baseline_g,  "rappel_pgd_0.05")
connue_pgd      = moy_par_famille(B_connue,    "rappel_pgd_0.05")
inconnue_pgd    = moy_par_famille(B_inconnue,  "rappel_pgd_0.05")

familles_g = sorted(familles_gen)
x = np.arange(len(familles_g))
largeur = 0.27

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(x - largeur, [baseline_pgd_g.get(f, 0) for f in familles_g],
       largeur, color=COULEUR_BASELINE,  alpha=0.85, label="Baseline")
ax.bar(x,           [connue_pgd.get(f, 0)      for f in familles_g],
       largeur, color=COULEUR_DEFENSE_B,  alpha=0.85, label="Défense B (famille connue)")
ax.bar(x + largeur, [inconnue_pgd.get(f, 0)    for f in familles_g],
       largeur, color=COULEUR_DEFENSE_B,  alpha=0.40, label="Défense B (famille inconnue)")

ax.set_xticks(x)
ax.set_xticklabels(familles_g, rotation=20, ha="right")
ax.set_ylim(0, 1.1)
ax.set_ylabel("Rappel moyen sous PGD (ε=0,05)")
ax.set_title("Généralisation leave-one-family-out\n(rappel moyen sur 5 graines par famille exclue)")
ax.legend(loc="upper right")

plt.tight_layout()
plt.savefig("figures/fig4_generalisation.png", bbox_inches="tight")
plt.close()
print("Figure 4 sauvegardée.")

# ---------------------------------------------------------------------------
# FIGURE 5 — Stabilité sur 15 graines (resultats_stabilite.csv)
# ---------------------------------------------------------------------------
# Rappel propre et rappel sous PGD, par graine, à 20 et 40 époques
# (agrégé toutes familles, défense B)

df_stab = pd.read_csv("resultats_stabilite.csv")

graines_20 = df_stab[df_stab["epoques"] == 20].groupby("graine")[["rappel_propre", "rappel_pgd_0.05"]].mean()
graines_40 = df_stab[df_stab["epoques"] == 40].groupby("graine")[["rappel_propre", "rappel_pgd_0.05"]].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, df_ep, titre in [
    (axes[0], graines_20, "20 époques"),
    (axes[1], graines_40, "40 époques"),
]:
    grainesX = df_ep.index.tolist()
    x = np.arange(len(grainesX))

    ax.plot(x, df_ep["rappel_propre"].values,  "o-", color=COULEUR_DEFENSE_B,
            linewidth=1.8, markersize=6, label="Rappel propre")
    ax.plot(x, df_ep["rappel_pgd_0.05"].values, "s--", color=COULEUR_PGD,
            linewidth=1.8, markersize=6, label="Rappel sous PGD")

    ax.set_xticks(x)
    ax.set_xticklabels([f"G{g}" for g in grainesX], fontsize=9)
    ax.set_ylim(0, 1.1)
    ax.set_xlabel("Graine")
    ax.set_title(f"Défense B — {titre}")
    ax.axhline(0.5, color="#ccc", linewidth=0.8, linestyle=":")
    ax.legend()

axes[0].set_ylabel("Rappel moyen (toutes familles)")
fig.suptitle("Stabilité de la Défense B sur 15 graines — comportement bimodal", fontsize=13)
plt.tight_layout()
plt.savefig("figures/fig5_stabilite_15graines.png", bbox_inches="tight")
plt.close()
print("Figure 5 sauvegardée.")

# ---------------------------------------------------------------------------
print("\nTerminé. Les 5 figures se trouvent dans le dossier  figures/")
print("  fig1_vulnerabilite_baseline.png")
print("  fig2_comparaison_defenses.png")
print("  fig3_rappel_par_famille.png")
print("  fig4_generalisation.png")
print("  fig5_stabilite_15graines.png")

Figure 1 sauvegardée.
Figure 2 sauvegardée.
Figure 3 sauvegardée.
Figure 4 sauvegardée.
Figure 5 sauvegardée.

Terminé. Les 5 figures se trouvent dans le dossier  figures/
  fig1_vulnerabilite_baseline.png
  fig2_comparaison_defenses.png
  fig3_rappel_par_famille.png
  fig4_generalisation.png
  fig5_stabilite_15graines.png
